# BreXpert-MM data audit

In [ ]:
from __future__ import annotations

import ast
import hashlib
import importlib.metadata
import json
import os
import platform
import re
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from tqdm.auto import tqdm

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."

# Edit this path on the remote server, or set BREXPERT_DATA_ROOT, if the data directory is elsewhere.
DATA_ROOT = Path(os.environ.get("BREXPERT_DATA_ROOT", REPO_ROOT.parent / "data")).resolve()
EXPORT_DIR = DATA_ROOT / "paper_audit"

# Use None for the full submission QC scan. Set an integer only for a quick dry run.
MAX_ARRAYS_PER_DATASET: int | None = None

sys.path.insert(0, str(REPO_ROOT))

from utils.vqa_findings_templates import FIELD_SPECS

KNOWN_FIELD_IDS = {spec.field_id for spec in FIELD_SPECS}
EXPECTED_IMAGE_SHAPES = {(512, 512), (1, 512, 512)}
EXPECTED_DTYPE = "uint16"
NOT_PRESENT = "not_present"

print(f"Repository root: {REPO_ROOT}")
print(f"Data root:       {DATA_ROOT}")
print(f"Export dir:      {EXPORT_DIR}")
print(f"Array scan:      {'full' if MAX_ARRAYS_PER_DATASET is None else MAX_ARRAYS_PER_DATASET}")

## Repository and environment snapshot

In [ ]:
def git_output(*args: str) -> str:
    result = subprocess.run(
        ["git", *args], cwd=REPO_ROOT, capture_output=True, text=True, check=False
    )
    return result.stdout.strip()


def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


pyproject_text = (REPO_ROOT / "pyproject.toml").read_text(encoding="utf-8")
makefile_text = (REPO_ROOT / "Makefile").read_text(encoding="utf-8")
requires_python = re.search(r'requires-python\s*=\s*"([^"]+)"', pyproject_text)
makefile_python = re.search(r'conda create --name \$\(PROJECT_NAME\) python=([^\s]+)', makefile_text)

environment_snapshot = pd.DataFrame(
    [
        ("git_commit", git_output("rev-parse", "HEAD")),
        ("git_dirty", str(bool(git_output("status", "--porcelain")))),
        ("python_runtime", sys.version.replace("\n", " ")),
        ("pyproject_requires_python", requires_python.group(1) if requires_python else "not found"),
        ("makefile_conda_python", makefile_python.group(1) if makefile_python else "not found"),
        ("platform", platform.platform()),
        ("machine", platform.machine()),
        ("processor", platform.processor()),
        ("cpu_count", str(os.cpu_count())),
        ("numpy", package_version("numpy")),
        ("pandas", package_version("pandas")),
        ("opencv-python", package_version("opencv-python")),
        ("pydicom", package_version("pydicom")),
        ("simpleitk", package_version("simpleitk")),
        ("scikit-image", package_version("scikit-image")),
    ],
    columns=["item", "value"],
)
display(environment_snapshot)


## Load reconstructed artifacts

In [ ]:
REQUIRED_COLUMNS = [
    "id",
    "patient",
    "dataset",
    "modality",
    "birads",
    "race",
    "machine",
    "exam",
    "segmentation",
    "context",
    "findings",
]


def is_missing(value: object) -> bool:
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip().lower() in {"", "nan", "none"}
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def parse_json_object(value: object) -> tuple[dict, str | None]:
    if is_missing(value):
        return {}, "missing"
    try:
        parsed = json.loads(str(value))
    except json.JSONDecodeError as exc:
        return {}, str(exc)
    if not isinstance(parsed, dict):
        return {}, f"expected object, got {type(parsed).__name__}"
    return parsed, None


def flatten_dict(value: object, parent: str = "") -> dict[str, object]:
    if isinstance(value, dict):
        out: dict[str, object] = {}
        for key, item in value.items():
            name = f"{parent}.{key}" if parent else str(key)
            out.update(flatten_dict(item, name))
        return out
    if isinstance(value, list):
        out = {}
        for index, item in enumerate(value):
            name = f"{parent}.{index}" if parent else str(index)
            out.update(flatten_dict(item, name))
        return out
    return {parent: value} if parent else {}


def parse_path_values(value: object) -> list[str]:
    if is_missing(value):
        return []
    if isinstance(value, (list, tuple)):
        return [str(item) for item in value if not is_missing(item)]
    text = str(value).strip()
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
        except (SyntaxError, ValueError):
            return [text]
        if isinstance(parsed, (list, tuple)):
            return [str(item) for item in parsed if not is_missing(item)]
    return [text]


def resolve_artifact_path(value: str) -> Path:
    path = Path(value)
    return path if path.is_absolute() else (REPO_ROOT / path).resolve()


def load_jsonl(path: Path) -> tuple[list[dict], list[dict]]:
    records: list[dict] = []
    errors: list[dict] = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                errors.append({"file": str(path), "line": line_number, "error": str(exc)})
                continue
            if not isinstance(record, dict):
                errors.append({"file": str(path), "line": line_number, "error": "record is not an object"})
                continue
            records.append(record)
    return records, errors


processed_files = sorted((DATA_ROOT / "processed").rglob("*.csv"))
assert processed_files, f"No processed CSV files found under {DATA_ROOT / 'processed'}"

processed_frames = []
for path in processed_files:
    frame = pd.read_csv(path, low_memory=False)
    frame["_source_csv"] = str(path)
    processed_frames.append(frame)
processed = pd.concat(processed_frames, ignore_index=True)

split_dir = DATA_ROOT / "report_generation_split"
split_files = [split_dir / f"all-rg-{split}.csv" for split in ["train", "val", "test"]]
split_frames = []
for path in split_files:
    if not path.exists():
        continue
    frame = pd.read_csv(path, low_memory=False)
    frame["split"] = path.stem.rsplit("-", 1)[-1]
    frame["_source_csv"] = str(path)
    split_frames.append(frame)
splits = pd.concat(split_frames, ignore_index=True) if split_frames else pd.DataFrame()

all_vqa_files = sorted(DATA_ROOT.glob("*_vqa_split/vqa_*.jsonl"))
canonical_vqa_files = [
    path for path in all_vqa_files if path.name in {"vqa_train.jsonl", "vqa_val.jsonl", "vqa_test.jsonl"}
]
vqa_records: list[dict] = []
vqa_json_errors: list[dict] = []
for path in canonical_vqa_files:
    records, errors = load_jsonl(path)
    variant = path.parent.name
    split = path.stem.removeprefix("vqa_")
    for record in records:
        record["_variant"] = variant
        record["_split_from_file"] = split
        record["_source_jsonl"] = str(path)
    vqa_records.extend(records)
    vqa_json_errors.extend(errors)

artifact_inventory = pd.DataFrame(
    [
        *[("processed_csv", str(path), path.stat().st_size) for path in processed_files],
        *[("split_csv", str(path), path.stat().st_size) for path in split_files if path.exists()],
        *[("vqa_jsonl", str(path), path.stat().st_size) for path in all_vqa_files],
    ],
    columns=["artifact_type", "path", "bytes"],
)
artifact_inventory["size_gib"] = artifact_inventory["bytes"] / (1024 ** 3)

print(f"Processed rows:       {len(processed):,}")
print(f"Split rows:           {len(splits):,}")
print(f"Canonical VQA rows:   {len(vqa_records):,}")
print(f"VQA JSON line errors: {len(vqa_json_errors):,}")
display(artifact_inventory)


## Canonical counts

In [ ]:
processed["_segmentation_count"] = processed["segmentation"].apply(parse_path_values).str.len()

source_counts = (
    processed.groupby("dataset", dropna=False)
    .agg(
        modalities=("modality", lambda values: ", ".join(sorted(set(values.dropna().astype(str))))),
        patients=("patient", "nunique"),
        images=("id", "size"),
        unique_image_ids=("id", "nunique"),
        images_with_segmentation=("_segmentation_count", lambda values: int((values > 0).sum())),
        segmentation_files=("_segmentation_count", "sum"),
    )
    .reset_index()
    .sort_values("dataset")
)
display(source_counts)

if not splits.empty:
    split_counts = (
        splits.groupby(["dataset", "modality", "split"], dropna=False)
        .agg(patients=("patient", "nunique"), images=("id", "size"))
        .reset_index()
        .sort_values(["dataset", "modality", "split"])
    )
    split_label_counts = (
        splits.groupby(["dataset", "modality", "split", "original_birads", "birads"], dropna=False)
        .agg(patients=("patient", "nunique"), images=("id", "size"))
        .reset_index()
        .sort_values(["dataset", "modality", "split", "original_birads", "birads"])
    )
    display(split_counts)
    display(split_label_counts)
else:
    split_counts = pd.DataFrame()
    split_label_counts = pd.DataFrame()
    display(Markdown("**No aggregate split CSV files were found.**"))


In [ ]:
split_lookup_columns = [
    column for column in ["id", "dataset", "modality", "patient", "original_birads", "birads", "findings"] if column in splits.columns
]
split_lookup = (
    splits[split_lookup_columns].drop_duplicates("id").rename(columns={"id": "source_id"})
    if not splits.empty
    else pd.DataFrame()
)

vqa_dialogue_rows = []
for record in vqa_records:
    conversation = record.get("conversation") if isinstance(record.get("conversation"), list) else []
    assistant_turns = [turn for turn in conversation if isinstance(turn, dict) and turn.get("role") == "assistant"]
    vqa_dialogue_rows.append(
        {
            "variant": record.get("_variant"),
            "split": record.get("_split_from_file"),
            "dialogue_id": str(record.get("id")),
            "source_id": str(record.get("source_id")),
            "questions": len(assistant_turns),
            "negative_questions": sum(bool(turn.get("is_negative", False)) for turn in assistant_turns),
        }
    )
vqa_dialogues = pd.DataFrame(vqa_dialogue_rows)

if not vqa_dialogues.empty and not split_lookup.empty:
    split_lookup["source_id"] = split_lookup["source_id"].astype(str)
    vqa_dialogues = vqa_dialogues.merge(split_lookup, on="source_id", how="left")
    vqa_counts = (
        vqa_dialogues.groupby(["variant", "dataset", "modality", "split"], dropna=False)
        .agg(dialogues=("dialogue_id", "size"), questions=("questions", "sum"), negative_questions=("negative_questions", "sum"))
        .reset_index()
        .sort_values(["variant", "dataset", "modality", "split"])
    )
    display(vqa_counts)
else:
    vqa_counts = pd.DataFrame()
    display(Markdown("**No canonical VQA files or split lookup were available for VQA counts.**"))


## Schema, field coverage, and metadata missingness

In [ ]:
schema_issue_rows = []
context_objects: list[dict] = []
findings_objects: list[dict] = []
context_field_rows = []
findings_field_rows = []

for dataset, group in processed.groupby("dataset", dropna=False):
    missing_columns = sorted(set(REQUIRED_COLUMNS) - set(group.columns))
    if missing_columns:
        schema_issue_rows.append({"dataset": dataset, "issue": "missing_required_columns", "count": len(missing_columns), "details": ", ".join(missing_columns)})
    schema_issue_rows.append({"dataset": dataset, "issue": "duplicate_id_rows", "count": int(group["id"].duplicated().sum()), "details": ""})
    schema_issue_rows.append({"dataset": dataset, "issue": "missing_patient_rows", "count": int(group["patient"].isna().sum()), "details": ""})
    schema_issue_rows.append({"dataset": dataset, "issue": "missing_exam_path_rows", "count": int(group["exam"].isna().sum()), "details": ""})

for row in processed.itertuples(index=False):
    context, context_error = parse_json_object(row.context)
    findings, findings_error = parse_json_object(row.findings)
    context_objects.append(context)
    findings_objects.append(findings)
    if context_error:
        schema_issue_rows.append({"dataset": row.dataset, "issue": "context_json_error", "count": 1, "details": context_error})
    if findings_error:
        schema_issue_rows.append({"dataset": row.dataset, "issue": "findings_json_error", "count": 1, "details": findings_error})
    for field, value in flatten_dict(context).items():
        context_field_rows.append({"dataset": row.dataset, "field": field, "value": value})
    for field, value in flatten_dict(findings).items():
        findings_field_rows.append({"dataset": row.dataset, "field": field, "value": value})

processed["context_obj"] = context_objects
processed["findings_obj"] = findings_objects

schema_issues = (
    pd.DataFrame(schema_issue_rows)
    .groupby(["dataset", "issue"], dropna=False)
    .agg(count=("count", "sum"), details=("details", lambda values: "; ".join(sorted(set(value for value in values if value)))))
    .reset_index()
    .sort_values(["dataset", "issue"])
)
display(schema_issues)

dataset_rows = processed.groupby("dataset").size().rename("dataset_rows")
context_fields = pd.DataFrame(context_field_rows)
findings_fields = pd.DataFrame(findings_field_rows)

def field_coverage_table(fields: pd.DataFrame) -> pd.DataFrame:
    if fields.empty:
        return pd.DataFrame()
    coverage = (
        fields.groupby(["dataset", "field"], dropna=False)
        .agg(present_rows=("value", "size"), unique_values=("value", lambda values: values.astype(str).nunique()))
        .reset_index()
    )
    coverage = coverage.merge(dataset_rows, on="dataset", how="left")
    coverage["coverage_percent"] = 100 * coverage["present_rows"] / coverage["dataset_rows"]
    return coverage.sort_values(["dataset", "field"])

context_field_coverage = field_coverage_table(context_fields)
findings_field_coverage = field_coverage_table(findings_fields)
display(context_field_coverage)
display(findings_field_coverage)


In [ ]:
def missing_or_unknown(value: object) -> bool:
    if is_missing(value):
        return True
    return str(value).strip().lower() in {"unknown", "not available", "not applicable"}


metadata_rows = []
for row in processed.itertuples(index=False):
    patient_context = row.context_obj.get("patient", {}) if isinstance(row.context_obj, dict) else {}
    metadata_rows.append(
        {
            "dataset": row.dataset,
            "race_source_field": row.race,
            "age_bin": patient_context.get("age"),
            "has_implants": patient_context.get("has implants"),
            "machine": row.machine,
        }
    )
metadata_values = pd.DataFrame(metadata_rows)

metadata_coverage_rows = []
for (dataset, field), values in metadata_values.set_index("dataset").stack().groupby(level=[0, 1]):
    values = values.reset_index(drop=True)
    missing_count = int(values.apply(missing_or_unknown).sum())
    metadata_coverage_rows.append(
        {
            "dataset": dataset,
            "field": field,
            "rows": len(values),
            "missing_or_unknown_rows": missing_count,
            "available_rows": len(values) - missing_count,
            "available_percent": 100 * (len(values) - missing_count) / len(values),
            "unique_nonmissing_values": values[~values.apply(missing_or_unknown)].astype(str).nunique(),
        }
    )
metadata_coverage = pd.DataFrame(metadata_coverage_rows).sort_values(["dataset", "field"])

metadata_value_counts = (
    metadata_values.melt(id_vars="dataset", var_name="field", value_name="value")
    .assign(value=lambda frame: frame["value"].fillna("<missing>").astype(str))
    .groupby(["dataset", "field", "value"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["dataset", "field", "rows"], ascending=[True, True, False])
)

unavailable_representation_fields = pd.DataFrame(
    [
        ("sex_or_gender", False, "not present in the common output schema"),
        ("provider_location", False, "not present in the common output schema"),
        ("race_ethnicity_definition", False, "source-specific definitions are not encoded in the outputs"),
    ],
    columns=["field", "available_from_outputs", "reason"],
)

display(metadata_coverage)
display(metadata_value_counts)
display(unavailable_representation_fields)


## Split leakage, completeness, and memmap alignment

In [ ]:
def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


if not splits.empty:
    overlap_rows = []
    for key in ["patient", "id", "exam"]:
        if key not in splits.columns:
            continue
        counts = splits.groupby(key, dropna=False)["split"].nunique()
        overlap_rows.append({"identifier": key, "values_in_multiple_splits": int((counts > 1).sum())})
    split_overlap = pd.DataFrame(overlap_rows)

    processed_ids = set(processed["id"].astype(str))
    split_ids = set(splits["id"].astype(str))
    split_completeness = pd.DataFrame(
        [
            ("processed_unique_ids", len(processed_ids)),
            ("split_unique_ids", len(split_ids)),
            ("processed_ids_missing_from_splits", len(processed_ids - split_ids)),
            ("split_ids_missing_from_processed", len(split_ids - processed_ids)),
        ],
        columns=["check", "count"],
    )

    patient_assignment = (
        splits[["patient", "split"]].drop_duplicates().sort_values(["patient", "split"]).astype(str)
    )
    patient_assignment_hash = sha256_text(patient_assignment.to_csv(index=False))
    split_manifest = pd.DataFrame(
        [(patient_assignment_hash, len(patient_assignment), git_output("rev-parse", "HEAD"))],
        columns=["patient_assignment_sha256", "patient_assignments", "git_commit"],
    )

    memmap_rows = []
    bytes_per_image = 512 * 512 * np.dtype(np.uint16).itemsize
    for split in ["train", "val", "test"]:
        csv_rows = int((splits["split"] == split).sum())
        path = split_dir / f"all-rg-{split}.bin"
        actual_bytes = path.stat().st_size if path.exists() else None
        expected_bytes = csv_rows * bytes_per_image
        memmap_rows.append(
            {
                "split": split,
                "csv_rows": csv_rows,
                "memmap_exists": path.exists(),
                "expected_bytes": expected_bytes,
                "actual_bytes": actual_bytes,
                "size_matches_csv": actual_bytes == expected_bytes if actual_bytes is not None else False,
            }
        )
    memmap_alignment = pd.DataFrame(memmap_rows)

    display(split_overlap)
    display(split_completeness)
    display(split_manifest)
    display(memmap_alignment)
else:
    split_overlap = pd.DataFrame()
    split_completeness = pd.DataFrame()
    split_manifest = pd.DataFrame()
    memmap_alignment = pd.DataFrame()
    display(Markdown("**No aggregate split files were found.**"))


## Image and segmentation QC

In [ ]:
image_inventory = processed[["dataset", "exam"]].dropna().drop_duplicates().rename(columns={"exam": "stored_path"})
image_inventory["kind"] = "image"

segmentation_rows = []
for row in processed[["dataset", "segmentation"]].itertuples(index=False):
    for path in parse_path_values(row.segmentation):
        segmentation_rows.append({"dataset": row.dataset, "stored_path": path, "kind": "segmentation"})
segmentation_inventory = pd.DataFrame(segmentation_rows).drop_duplicates() if segmentation_rows else pd.DataFrame(columns=["dataset", "stored_path", "kind"])

array_inventory = pd.concat([image_inventory, segmentation_inventory], ignore_index=True)
array_inventory["resolved_path"] = array_inventory["stored_path"].astype(str).apply(lambda value: str(resolve_artifact_path(value)))
array_inventory["exists"] = array_inventory["resolved_path"].apply(lambda value: Path(value).exists())


def inspect_arrays(inventory: pd.DataFrame) -> pd.DataFrame:
    result_rows = []
    for (dataset, kind), group in inventory.groupby(["dataset", "kind"], dropna=False):
        selected = group.sort_values("resolved_path")
        if MAX_ARRAYS_PER_DATASET is not None:
            selected = selected.head(MAX_ARRAYS_PER_DATASET)
        for row in tqdm(selected.itertuples(index=False), total=len(selected), desc=f"{dataset} {kind}"):
            result = {
                "dataset": dataset,
                "kind": kind,
                "stored_path": row.stored_path,
                "resolved_path": row.resolved_path,
                "exists": row.exists,
                "readable": False,
                "shape": None,
                "dtype": None,
                "minimum": None,
                "maximum": None,
                "constant_array": None,
                "nonzero_pixels": None,
                "error": None,
            }
            if not row.exists:
                result["error"] = "missing file"
                result_rows.append(result)
                continue
            try:
                array = np.load(row.resolved_path, mmap_mode="r")
                minimum = int(array.min())
                maximum = int(array.max())
                result.update(
                    {
                        "readable": True,
                        "shape": str(tuple(array.shape)),
                        "dtype": str(array.dtype),
                        "minimum": minimum,
                        "maximum": maximum,
                        "constant_array": minimum == maximum,
                        "nonzero_pixels": int(np.count_nonzero(array)),
                    }
                )
            except Exception as exc:
                result["error"] = f"{type(exc).__name__}: {exc}"
            result_rows.append(result)
    return pd.DataFrame(result_rows)


array_qc = inspect_arrays(array_inventory)
if not array_qc.empty:
    array_qc["expected_shape"] = array_qc["shape"].apply(
        lambda value: value in {str(shape) for shape in EXPECTED_IMAGE_SHAPES} if value is not None else False
    )
    array_qc["expected_dtype"] = array_qc["dtype"] == EXPECTED_DTYPE
    array_qc["unexpected_shape"] = array_qc["readable"] & ~array_qc["expected_shape"]
    array_qc["unexpected_dtype"] = array_qc["readable"] & ~array_qc["expected_dtype"]
    array_qc["empty_segmentation"] = (array_qc["kind"] == "segmentation") & (array_qc["nonzero_pixels"] == 0)
    array_qc_summary = (
        array_qc.groupby(["dataset", "kind"], dropna=False)
        .agg(
            checked=("resolved_path", "size"),
            missing_files=("exists", lambda values: int((~values).sum())),
            unreadable_files=("readable", lambda values: int((~values).sum())),
            unexpected_shapes=("unexpected_shape", "sum"),
            unexpected_dtypes=("unexpected_dtype", "sum"),
            constant_arrays=("constant_array", lambda values: int(values.fillna(False).sum())),
            empty_segmentations=("empty_segmentation", "sum"),
        )
        .reset_index()
        .sort_values(["dataset", "kind"])
    )
else:
    array_qc_summary = pd.DataFrame()

display(array_qc_summary)


## VQA structural QC and lesion-status validation

In [ ]:
def expected_answers(findings: dict) -> dict[str, str]:
    answers = {
        field: str(value)
        for field, value in flatten_dict(findings).items()
        if value != NOT_PRESENT
    }
    if "lesion" in findings:
        answers["lesion.presence"] = "no" if findings["lesion"] == NOT_PRESENT else "yes"
    return answers


def expected_lesion_status(findings: dict) -> str:
    if "lesion" not in findings:
        return "unknown"
    return "absent" if findings["lesion"] == NOT_PRESENT else "present"


source_records = {}
if not splits.empty:
    for row in splits.itertuples(index=False):
        findings, _ = parse_json_object(row.findings)
        source_records[str(row.id)] = {
            "dataset": row.dataset,
            "modality": row.modality,
            "split": row.split,
            "findings": findings,
        }

vqa_issue_rows = [{**error, "issue": "json_line_error"} for error in vqa_json_errors]
vqa_turn_rows = []
lesion_status_rows = []

for record in vqa_records:
    source_id = str(record.get("source_id"))
    dialogue_id = str(record.get("id"))
    variant = record.get("_variant")
    file_split = record.get("_split_from_file")
    source = source_records.get(source_id)
    if source is None:
        vqa_issue_rows.append({"variant": variant, "dialogue_id": dialogue_id, "issue": "source_id_not_found"})
        continue
    if record.get("split") != file_split or source["split"] != file_split:
        vqa_issue_rows.append({"variant": variant, "dialogue_id": dialogue_id, "issue": "split_mismatch"})

    conversation = record.get("conversation")
    if not isinstance(conversation, list):
        vqa_issue_rows.append({"variant": variant, "dialogue_id": dialogue_id, "issue": "conversation_not_list"})
        continue
    if len(conversation) % 2 != 0:
        vqa_issue_rows.append({"variant": variant, "dialogue_id": dialogue_id, "issue": "odd_conversation_length"})

    assistant_turns = [turn for turn in conversation if isinstance(turn, dict) and turn.get("role") == "assistant"]
    metadata = record.get("metadata") if isinstance(record.get("metadata"), dict) else {}
    negative_count = sum(bool(turn.get("is_negative", False)) for turn in assistant_turns)
    if metadata.get("question_count") != len(assistant_turns):
        vqa_issue_rows.append({"variant": variant, "dialogue_id": dialogue_id, "issue": "question_count_mismatch"})
    if metadata.get("negative_count") != negative_count:
        vqa_issue_rows.append({"variant": variant, "dialogue_id": dialogue_id, "issue": "negative_count_mismatch"})
    if metadata.get("has_negative") != (negative_count > 0):
        vqa_issue_rows.append({"variant": variant, "dialogue_id": dialogue_id, "issue": "has_negative_mismatch"})

    expected_status = expected_lesion_status(source["findings"])
    observed_status = metadata.get("lesion_status")
    lesion_status_rows.append(
        {
            "variant": variant,
            "dataset": source["dataset"],
            "modality": source["modality"],
            "split": file_split,
            "expected_lesion_status": expected_status,
            "observed_lesion_status": observed_status,
            "matches": expected_status == observed_status,
        }
    )

    answers = expected_answers(source["findings"])
    for pair_index in range(0, len(conversation) - 1, 2):
        question = conversation[pair_index]
        answer = conversation[pair_index + 1]
        if not isinstance(question, dict) or not isinstance(answer, dict):
            vqa_issue_rows.append({"variant": variant, "dialogue_id": dialogue_id, "issue": "turn_not_object"})
            continue
        if question.get("role") != "user" or answer.get("role") != "assistant":
            vqa_issue_rows.append({"variant": variant, "dialogue_id": dialogue_id, "issue": "role_order_mismatch"})
        field_id = answer.get("field_id")
        is_negative = bool(answer.get("is_negative", False))
        if field_id not in KNOWN_FIELD_IDS:
            vqa_issue_rows.append({"variant": variant, "dialogue_id": dialogue_id, "issue": "unknown_field_id"})
        if not is_negative and field_id not in answers:
            vqa_issue_rows.append({"variant": variant, "dialogue_id": dialogue_id, "issue": "nonnegative_field_not_in_findings"})
        if not is_negative and field_id in answers and str(answer.get("text")) != answers[field_id]:
            vqa_issue_rows.append({"variant": variant, "dialogue_id": dialogue_id, "issue": "answer_mismatch"})
        vqa_turn_rows.append(
            {
                "variant": variant,
                "dataset": source["dataset"],
                "modality": source["modality"],
                "split": file_split,
                "dialogue_id": dialogue_id,
                "source_id": source_id,
                "field_id": field_id,
                "task_type": answer.get("task_type"),
                "is_negative": is_negative,
            }
        )

vqa_issues = pd.DataFrame(vqa_issue_rows)
if not vqa_issues.empty and "issue" in vqa_issues.columns:
    vqa_issue_summary = vqa_issues.groupby("issue", dropna=False).size().reset_index(name="count").sort_values("count", ascending=False)
else:
    vqa_issue_summary = pd.DataFrame(columns=["issue", "count"])

vqa_turns = pd.DataFrame(vqa_turn_rows)
if not vqa_turns.empty:
    vqa_field_counts = (
        vqa_turns.groupby(["variant", "dataset", "modality", "split", "field_id", "is_negative"], dropna=False)
        .size()
        .reset_index(name="questions")
        .sort_values(["variant", "dataset", "modality", "split", "field_id", "is_negative"])
    )
else:
    vqa_field_counts = pd.DataFrame()

lesion_status_validation = pd.DataFrame(lesion_status_rows)
if not lesion_status_validation.empty:
    lesion_status_summary = (
        lesion_status_validation.groupby(
            ["variant", "dataset", "expected_lesion_status", "observed_lesion_status"], dropna=False
        )
        .agg(dialogues=("matches", "size"), mismatches=("matches", lambda values: int((~values).sum())))
        .reset_index()
        .sort_values(["variant", "dataset", "expected_lesion_status", "observed_lesion_status"])
    )
else:
    lesion_status_summary = pd.DataFrame()

display(vqa_issue_summary)
display(vqa_field_counts)
display(lesion_status_summary)


## Automated privacy and free-text screening

In [ ]:
controlled_value_rows = []
for row in processed.itertuples(index=False):
    controlled_value_rows.append({"dataset": row.dataset, "field": "race", "value": row.race})
    controlled_value_rows.append({"dataset": row.dataset, "field": "machine", "value": row.machine})
    for field, value in flatten_dict(row.context_obj).items():
        controlled_value_rows.append({"dataset": row.dataset, "field": f"context.{field}", "value": value})
    for field, value in flatten_dict(row.findings_obj).items():
        controlled_value_rows.append({"dataset": row.dataset, "field": f"findings.{field}", "value": value})

controlled_values = pd.DataFrame(controlled_value_rows)
controlled_values["value"] = controlled_values["value"].fillna("").astype(str)

patterns = {
    "email_like": re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),
    "url_like": re.compile(r"\b(?:https?://|www\.)\S+", re.IGNORECASE),
    "newline": re.compile(r"[\r\n]"),
}

privacy_pattern_rows = []
for (dataset, field), group in controlled_values.groupby(["dataset", "field"], dropna=False):
    for pattern_name, pattern in patterns.items():
        privacy_pattern_rows.append(
            {
                "dataset": dataset,
                "field": field,
                "pattern": pattern_name,
                "matching_rows": int(group["value"].str.contains(pattern).sum()),
            }
        )
privacy_pattern_screen = pd.DataFrame(privacy_pattern_rows)
privacy_pattern_screen = privacy_pattern_screen[privacy_pattern_screen["matching_rows"] > 0].sort_values(["dataset", "field", "pattern"])

free_text_profile = (
    controlled_values.groupby(["dataset", "field"], dropna=False)
    .agg(rows=("value", "size"), unique_values=("value", "nunique"), max_length=("value", lambda values: values.str.len().max()))
    .reset_index()
    .sort_values(["dataset", "field"])
)

path_screen_rows = []
for row in processed[["dataset", "id", "patient", "exam", "segmentation"]].itertuples(index=False):
    for field in ["id", "patient", "exam", "segmentation"]:
        for value in parse_path_values(getattr(row, field)) if field == "segmentation" else [str(getattr(row, field))]:
            path_screen_rows.append(
                {
                    "dataset": row.dataset,
                    "field": field,
                    "is_absolute_path": Path(value).is_absolute() if field in {"exam", "segmentation"} else False,
                    "references_raw_directory": "/raw/" in value.replace("\\", "/"),
                }
            )
path_screen = (
    pd.DataFrame(path_screen_rows)
    .groupby(["dataset", "field"], dropna=False)
    .agg(rows=("field", "size"), absolute_path_rows=("is_absolute_path", "sum"), raw_directory_reference_rows=("references_raw_directory", "sum"))
    .reset_index()
    .sort_values(["dataset", "field"])
)

display(privacy_pattern_screen)
display(free_text_profile)
display(path_screen)


## Storage footprint and key artifact checksums

In [ ]:
def directory_size(path: Path) -> tuple[int, int]:
    if not path.exists():
        return 0, 0
    file_count = 0
    byte_count = 0
    for item in path.rglob("*"):
        if item.is_file():
            file_count += 1
            byte_count += item.stat().st_size
    return file_count, byte_count


storage_rows = []
for name in ["raw", "processed", "report_generation_split", "base_vqa_split", "robust_vqa_split"]:
    path = DATA_ROOT / name
    file_count, byte_count = directory_size(path)
    storage_rows.append(
        {"directory": name, "exists": path.exists(), "files": file_count, "bytes": byte_count, "size_gib": byte_count / (1024 ** 3)}
    )
storage_summary = pd.DataFrame(storage_rows)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest_files = [REPO_ROOT / "Makefile", REPO_ROOT / "pyproject.toml", *processed_files, *[path for path in split_files if path.exists()], *canonical_vqa_files]
artifact_manifest_rows = []
for path in tqdm(manifest_files, desc="Hashing key artifacts"):
    artifact_manifest_rows.append(
        {
            "path": str(path),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
    )
artifact_manifest = pd.DataFrame(artifact_manifest_rows).sort_values("path")

display(storage_summary)
display(artifact_manifest)


## TODO coverage and limitations

Separates evidence produced here from TODOs needing code changes, timed reruns, raw-source analysis, or non-code work. Notably: inclusion/exclusion flow and primary-finding affected counts are not recoverable from final CSVs; runtime needs a clean timed rerun; this is QC evidence, not a replacement for CI tests.

In [ ]:
todo_coverage = pd.DataFrame(
    [
        ("Canonical patient/image/segmentation counts", "addressed", "source_counts"),
        ("Dialogue and question counts", "addressed when canonical VQA files exist", "vqa_counts"),
        ("Counts by source, modality, split, and label", "addressed", "split_counts; split_label_counts; vqa_field_counts"),
        ("Demographic metadata coverage and missingness", "partially addressed", "metadata_coverage; metadata_value_counts"),
        ("Sex/gender definitions and provider locations", "not derivable from outputs", "unavailable_representation_fields"),
        ("Storage and software environment", "addressed", "storage_summary; environment_snapshot"),
        ("Pipeline runtime and clean-machine reconstruction", "requires timed rerun", "not present in current artifacts"),
        ("Schema, image, segmentation, leakage, and VQA QC", "addressed as notebook evidence", "schema_issues; array_qc_summary; split_overlap; vqa_issue_summary"),
        ("Automated privacy/free-text screen", "partially addressed", "privacy_pattern_screen; free_text_profile; path_screen"),
        ("Independent de-identification audit", "requires separate review", "not resolved by pattern matching"),
        ("Lesion-status metadata validation", "addressed", "lesion_status_summary"),
        ("Current split manifest and key checksums", "addressed", "split_manifest; artifact_manifest"),
        ("Deterministic identifiers and split implementation", "requires code changes", "not fixed by this notebook"),
        ("Inclusion/exclusion flow counts", "not derivable from final artifacts", "requires adapter instrumentation or raw-source analysis"),
        ("Primary-finding selection affected-record count", "not derivable from final artifacts", "requires adapter instrumentation or raw-source analysis"),
        ("Source-version manifest", "partially addressed", "artifact_manifest does not identify upstream source versions"),
        ("Expert QA review and agreement", "requires human review", "not addressed"),
        ("Model baselines and metrics", "requires experiments", "not addressed"),
        ("Licensing, ethics, legal, authorship, conflicts, funding, contacts", "requires non-code evidence", "not addressed"),
    ],
    columns=["paper_todo", "status", "evidence_or_reason"],
)
display(todo_coverage)


## Export paper-ready tables

In [ ]:
tables = {
    # "environment_snapshot": environment_snapshot,
    "artifact_inventory": artifact_inventory,
    "source_counts": source_counts,
    "split_counts": split_counts,
    "split_label_counts": split_label_counts,
    "vqa_counts": vqa_counts,
    "schema_issues": schema_issues,
    "context_field_coverage": context_field_coverage,
    "findings_field_coverage": findings_field_coverage,
    # "metadata_coverage": metadata_coverage,
    # "metadata_value_counts": metadata_value_counts,
    # "unavailable_representation_fields": unavailable_representation_fields,
    "split_overlap": split_overlap,
    "split_completeness": split_completeness,
    # "split_manifest": split_manifest,
    # "memmap_alignment": memmap_alignment,
    "array_qc_summary": array_qc_summary,
    "vqa_issue_summary": vqa_issue_summary,
    "vqa_field_counts": vqa_field_counts,
    "lesion_status_summary": lesion_status_summary,
    "privacy_pattern_screen": privacy_pattern_screen,
    "free_text_profile": free_text_profile,
    "path_screen": path_screen,
    "storage_summary": storage_summary,
    "artifact_manifest": artifact_manifest,
    "todo_coverage": todo_coverage,
}

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for name, table in tables.items():
    table.to_csv(EXPORT_DIR / f"{name}.csv", index=False)

print(f"Exported {len(tables)} tables to {EXPORT_DIR}")
